# `programming-language-identification-100plus-lite` — ONNX Runtime

Same model as the PyTorch demo, exported to ONNX (opset 20). No torch needed at inference time. CPU-friendly: ~57 texts/sec single-thread on commodity hardware (2.37× philomath-1209 on the same box).

Run end-to-end in Colab or Jupyter.

## Install dependencies

In [ ]:
%%capture
!pip install -q -U onnxruntime huggingface_hub numpy


## Download ONNX model + label index

In [ ]:
import json
import numpy as np
import onnxruntime as ort
from huggingface_hub import hf_hub_download

REPO = 'FrameByFrame/programming-language-identification-100plus-lite'
onnx_path = hf_hub_download(REPO, 'onnx/model.onnx')
# external weight blob lives next to the .onnx file
hf_hub_download(REPO, 'onnx/model.onnx.data')
lang2idx = json.loads(open(hf_hub_download(REPO, 'onnx/lang2idx.json')).read())
meta = json.loads(open(hf_hub_download(REPO, 'onnx/onnx_metadata.json')).read())

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
idx2lang = {v: k for k, v in lang2idx.items()}
MAX_LEN = meta['max_len']
print(f'{len(idx2lang)} labels | max_len={MAX_LEN} | providers={sess.get_providers()}')

## Helpers

In [ ]:
def encode(texts, max_len=MAX_LEN):
    out = np.full((len(texts), max_len), 256, dtype=np.int64)
    for i, t in enumerate(texts):
        b = t.encode('utf-8', errors='replace')[:max_len]
        out[i, :len(b)] = np.frombuffer(b, dtype=np.uint8)
    return out


def softmax(logits, axis=-1):
    e = np.exp(logits - logits.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def predict(texts, top_k=3):
    logits = sess.run(None, {'byte_ids': encode(texts)})[0]
    probs = softmax(logits)
    top_i = np.argsort(-probs, axis=-1)[:, :top_k]
    return [[(idx2lang[int(j)], float(probs[r, j])) for j in row]
            for r, row in enumerate(top_i)]

## Predict

In [ ]:
samples = [
    "def fib(n):\n    return n if n < 2 else fib(n-1) + fib(n-2)",
    "fn main() {\n    println!(\"hello, world\");\n}",
    "package main\nimport \"fmt\"\nfunc main() { fmt.Println(\"hi\") }",
    "#include <stdio.h>\nint main() { printf(\"hi\\n\"); return 0; }",
    "SELECT name FROM users WHERE id = 42;",
]
for text, top in zip(samples, predict(samples)):
    print(f'{top[0][0]:<14s}  {top[0][1]:.3f}   ({top[1][0]} {top[1][1]:.2f}, {top[2][0]} {top[2][1]:.2f})  | {text[:60]!r}')

## Throughput sanity check

In [ ]:
import time
warm = encode(samples * 13)[:64]
for _ in range(3):
    sess.run(None, {'byte_ids': warm})
t0 = time.time()
for _ in range(40):
    sess.run(None, {'byte_ids': warm})
elapsed = time.time() - t0
print(f'{40*64/elapsed:.0f} texts/sec  ({elapsed:.2f}s for 40 batches of 64)')